## Final Workflow: ML based integration pipeline (Sorted Neighbourhood Blocker)

In [1]:
from pathlib import Path

ROOT = Path.cwd()

DATA_DIR = ROOT / "parquet"
OUTPUT_DIR = ROOT / "output"
MLDS_DIR = ROOT / "ml-datasets"
BLOCK_EVAL_DIR = OUTPUT_DIR / "blocking_evaluation"
CORR_DIR = OUTPUT_DIR / "correspondences"

PIPELINE_DIR = OUTPUT_DIR / "data_fusion"
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

### Schema Mapping

In [2]:
from PyDI.io import load_parquet, load_csv

amazon_books = load_parquet(
    DATA_DIR / "amazon.parquet",
    name="amazon_books"
)

goodreads = load_parquet(
    DATA_DIR / "goodreads.parquet",
    name="goodreads"
)

metabooks = load_parquet(
  DATA_DIR / "metabooks.parquet",
  name="metabooks"
)

In [6]:
goodreads

,id,title,author,rating,numratings,language,genres,bookformat,edition,page_count,publisher,publish_year,price,isbn_clean
0,goodreads_1,the hunger games,suzanne collins,4.33,6376780,English,"['Young Adult', 'Fiction', 'Dystopia', 'Fantas...",Hardcover,First Edition,374,Scholastic Press,2008,5.09,0439023483
1,goodreads_2,harry potter and the order of the phoenix,jk rowling mary grandpr illustrator,4.50,2507623,English,"['Fantasy', 'Young Adult', 'Fiction', 'Magic',...",Paperback,US Edition,870,Scholastic Inc.,2004,7.38,0439358078
2,goodreads_3,to kill a mockingbird,harper lee,4.28,4501075,English,"['Classics', 'Fiction', 'Historical Fiction', ...",Paperback,nan,324,Harper Perennial Modern Classics,2006,NaN,nan
3,goodreads_4,pride and prejudice,jane austen anna quindlen introduction,4.26,2998241,English,"['Classics', 'Fiction', 'Romance', 'Historical...",Paperback,"Modern Library Classics, USA / CAN",279,Modern Library,2000,NaN,nan
4,goodreads_5,twilight,stephenie meyer,3.60,4964519,English,"['Young Adult', 'Fantasy', 'Romance', 'Vampire...",Paperback,nan,501,"Little, Brown and Company",2006,2.10,0316015849
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52473,goodreads_52474,fractured,cheri schmidt goodreads author,4.00,871,English,"['Vampires', 'Paranormal', 'Young Adult', 'Rom...",Nook,nan,<NA>,Cheri Schmidt,2011,NaN,nan
52474,goodreads_52475,anasazi,emma michaels,4.19,37,English,"['Mystery', 'Young Adult']",Paperback,First Edition,190,Bokheim Publishing,2011,NaN,nan
52475,goodreads_52476,marked,kim richardson goodreads author,3.70,6674,English,"['Fantasy', 'Young Adult', 'Paranormal', 'Ange...",Paperback,nan,280,CreateSpace,2011,7.37,1461017092
52476,goodreads_52477,wayward son,tom pollack goodreads author john loftus goodr...,3.85,238,English,"['Fiction', 'Mystery', 'Historical Fiction', '...",Paperback,1st edition,507,Cascada Productions,2011,2.86,1450755631


In [7]:
from rapidfuzz import fuzz
from sentence_transformers import SentenceTransformer, util


# embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

def similarity_score(src, tgt):
    """Compute semantic + fuzzy similarity between column names."""
    emb_src = model.encode(src, convert_to_tensor=True)
    emb_tgt = model.encode(tgt, convert_to_tensor=True)
    semantic_sim = util.cos_sim(emb_src, emb_tgt).item()
    fuzzy_sim = fuzz.token_sort_ratio(src, tgt) / 100.0
    return 0.6 * semantic_sim + 0.4 * fuzzy_sim


def schema_match_semantic(source_cols, target_cols, threshold=0.6):
    """Find best target column for each source column."""
    mapping = {}
    for src in source_cols:
        best_tgt, best_score = None, 0
        for tgt in target_cols:
            score = similarity_score(src, tgt)
            if score > best_score:
                best_score, best_tgt = score, tgt
        mapping[src] = best_tgt if best_score >= threshold else None
    return mapping

In [8]:
def unify_schemas(schema_dict, target_schema_name="metabooks"):
    """
    Use one dataset (e.g., 'metabooks') as target schema.
    Match all other schemas to it.
    """
    if target_schema_name not in schema_dict:
        raise ValueError(f"Target schema '{target_schema_name}' not found in schema_dict")

    target_cols = schema_dict[target_schema_name]
    mappings = {}

    for name, cols in schema_dict.items():
        if name == target_schema_name:
            # Target schema maps to itself
            mappings[name] = {col: col for col in cols}
        else:
            mappings[name] = schema_match_semantic(cols, target_cols)

    return target_cols, mappings

In [9]:
schemas = {
    "amazon": amazon_books.columns,
    "metabooks": metabooks.columns,
    "goodreads": goodreads.columns
}

target_schema, mappings = unify_schemas(schemas, target_schema_name="goodreads")

print("Target Schema:")
print(target_schema)

print("\nSchema Mappings:")
for name, mapping in mappings.items():
    print(f"\n{name}:")
    for k, v in mapping.items():
        print(f"  {k:20} -> {v}")

Target Schema:
Index(['id', 'title', 'author', 'rating', 'numratings', 'language', 'genres',
       'bookformat', 'edition', 'page_count', 'publisher', 'publish_year',
       'price', 'isbn_clean'],
      dtype='object')

Schema Mappings:

amazon:
  id                   -> id
  title                -> title
  book-author          -> author
  year-of-publication  -> publish_year
  publisher            -> publisher
  isbn_clean           -> isbn_clean

metabooks:
  id                   -> id
  title                -> title
  author_name          -> author
  average_rating       -> rating
  num_rating           -> numratings
  language             -> language
  genres               -> genres
  publisher            -> publisher
  page_count           -> page_count
  price                -> price
  publish_year         -> publish_year
  isbn_clean           -> isbn_clean

goodreads:
  id                   -> id
  title                -> title
  author               -> author
  rating       

In [10]:
def apply_schema_mapping(df, mapping):
    rename_dict = {k: v for k, v in mapping.items() if v is not None}
    return df.rename(columns=rename_dict)

amazon_books = apply_schema_mapping(amazon_books, mappings["amazon"])
metabooks = apply_schema_mapping(metabooks, mappings["metabooks"])
goodreads = apply_schema_mapping(goodreads, mappings["goodreads"])

In [17]:
display(goodreads.columns)
display(metabooks.columns)

Index(['id', 'title', 'author', 'rating', 'numratings', 'language', 'genres',
       'bookformat', 'edition', 'page_count', 'publisher', 'publish_year',
       'price', 'isbn_clean'],
      dtype='object')

Index(['id', 'title', 'author', 'rating', 'numratings', 'language', 'genres',
       'publisher', 'page_count', 'price', 'publish_year', 'isbn_clean'],
      dtype='object')

### Entity Matching

In [56]:
train_m2a = load_csv(
    MLDS_DIR / "train_MA.csv",
    name="train_metabooks_amazon",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

test_m2a = load_csv(
    MLDS_DIR / "test_MA.csv",
    name="test_metabooks_amazon",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

train_m2g = load_csv(
    MLDS_DIR / "train_MG.csv",
    name="train_metabooks_goodreads",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

test_m2g = load_csv(
    MLDS_DIR / "test_MG.csv",
    name="test_metabooks_goodreads",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

In [57]:
from PyDI.entitymatching import StringComparator, NumericComparator

comparators_m2a = [
    StringComparator(column='title',similarity_function='jaccard'),
    StringComparator(column='title',similarity_function='cosine'),
    StringComparator(column='title',similarity_function='levenshtein'),
    StringComparator(column='title',similarity_function='jaro_winkler'),
    
    StringComparator(column='author',similarity_function='jaccard', preprocess=str.lower),
    StringComparator(column='author',similarity_function='cosine', preprocess=str.lower),
    StringComparator(column='author',similarity_function='levenshtein', preprocess=str.lower),

    StringComparator(column='publisher',similarity_function='jaccard', preprocess=str.lower),
    StringComparator(column='publisher',similarity_function='cosine', preprocess=str.lower),
    StringComparator(column='publisher',similarity_function='levenshtein', preprocess=str.lower),
    
    StringComparator(column='isbn_clean',similarity_function='cosine', preprocess=str.lower),
    StringComparator(column='isbn_clean',similarity_function='levenshtein', preprocess=str.lower),

    NumericComparator(column='publish_year',max_difference=1)
]

comparators_m2g = comparators_m2a + [
    StringComparator(column='genres',
                     similarity_function='jaccard',
                     preprocess=str.lower, 
                     list_strategy='concatenate'),

    StringComparator(column='genres',
                     similarity_function='jaccard',
                     preprocess=str.lower,
                     list_strategy='best_match'),

    NumericComparator(column="price", max_difference=5),
    NumericComparator(column="page_count", max_difference=10),
    NumericComparator(column="rating", max_difference=0.2)
]

### Sorted Neighbourhood Blocker

In [58]:
from PyDI.entitymatching import SortedNeighbourhoodBlocker


blocker_m2a = SortedNeighbourhoodBlocker(
    metabooks, amazon_books,
    key='title',
    window=20,
    batch_size=500,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)


blocker_m2g = SortedNeighbourhoodBlocker(
    metabooks, goodreads,
    key='title',
    window=20,
    batch_size=500,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

### ML-based Matcher

In [59]:
from PyDI.entitymatching import FeatureExtractor

feature_extractor_m2a = FeatureExtractor(comparators_m2a)
feature_extractor_m2g = FeatureExtractor(comparators_m2g)

train_features_m2a = feature_extractor_m2a.create_features(
    metabooks, amazon_books, train_m2a[['id1', 'id2']], labels=train_m2a['label'], id_column='id'
)

train_features_m2g = feature_extractor_m2g.create_features(
    metabooks, goodreads, train_m2g[['id1', 'id2']], labels=train_m2g['label'], id_column='id'
)

feature_columns_m2a = [col for col in train_features_m2a.columns if col not in ['id1', 'id2', 'label']]
X_train_m2a = train_features_m2a[feature_columns_m2a]
y_train_m2a = train_features_m2a['label']

feature_columns_m2g = [col for col in train_features_m2g.columns if col not in ['id1', 'id2', 'label']]
X_train_m2g = train_features_m2g[feature_columns_m2g]
y_train_m2g = train_features_m2g['label']

training_datasets = [(X_train_m2a, y_train_m2a),(X_train_m2g, y_train_m2g)]

In [60]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, f1_score
import pandas as pd

# classifiers
classifiers = {
    'RandomForestClassifier': RandomForestClassifier(random_state=42),
    'GradientBoostingClassifier': GradientBoostingClassifier(random_state=42),
    'SVC': SVC(probability=True, random_state=42),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42)
}

# Define parameter grids
param_grids = {
    'RandomForestClassifier': {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'class_weight': ['balanced', None],
        'min_samples_split': [2, 5]
    },
    'GradientBoostingClassifier': {
        'n_estimators': [100, 200],
        'learning_rate': [0.05, 0.1],
        'max_depth': [3, 5]
    },
    'SVC': {
        'C': [0.1, 1, 10],
        'kernel': ['linear', 'rbf'],
        'gamma': ['scale', 'auto']
    },
    'LogisticRegression': {
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l2'],
        'solver': ['lbfgs', 'liblinear']
    }
}


scorer = make_scorer(f1_score)
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best_models = []


for data in training_datasets:
    grid_search_results = {}
    best_overall_score = -1
    best_overall_model = None
    best_model_name = None

    for name, model in classifiers.items():
        print(f"Running GridSearchCV for {name}...")
        
        grid = GridSearchCV(
            estimator=model,
            param_grid=param_grids[name],
            scoring=scorer,
            cv=cv_folds,
            n_jobs=-1,
            verbose=0
        )
        
        grid.fit(data[0], data[1])
        
        grid_search_results[name] = {
            'grid_search': grid,
            'best_score': grid.best_score_,
            'best_params': grid.best_params_,
            'best_estimator': grid.best_estimator_
        }

        if grid.best_score_ > best_overall_score:
            best_overall_model = grid.best_estimator_

    best_models.append(best_overall_model)

Running GridSearchCV for RandomForestClassifier...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
huggingface/tokenizers: The current process just got f

Running GridSearchCV for GradientBoostingClassifier...
Running GridSearchCV for SVC...
Running GridSearchCV for LogisticRegression...
Running GridSearchCV for RandomForestClassifier...
Running GridSearchCV for GradientBoostingClassifier...
Running GridSearchCV for SVC...
Running GridSearchCV for LogisticRegression...


In [13]:
from PyDI.entitymatching import MLBasedMatcher


ml_matcher_m2a = MLBasedMatcher(feature_extractor_m2a)
ml_matcher_m2g = MLBasedMatcher(feature_extractor_m2g)

correspondences_m2a = ml_matcher_m2a.match(
    metabooks, amazon_books,
    candidates=blocker_m2a,
    id_column='id',
    trained_classifier=best_models[0]
)

correspondences_m2g = ml_matcher_m2g.match(
    metabooks, goodreads,
    candidates=blocker_m2g,
    id_column='id',
    trained_classifier=best_models[1]
)

In [14]:
correspondences_m2a.to_csv(CORR_DIR / "workflow_corr_m2a.csv", index=False)
correspondences_m2g.to_csv(CORR_DIR / "workflow_corr_m2g.csv", index=False)

### Data Fusion

In [18]:
from PyDI.fusion import DataFusionStrategy, DataFusionEngine, longest_string, union, prefer_higher_trust
import pandas as pd

amazon_books.attrs["trust_score"] = 3
metabooks.attrs["trust_score"] = 2
goodreads.attrs["trust_score"] = 1

correspondences_m2a = pd.read_csv(CORR_DIR / "workflow_corr_m2a.csv")
correspondences_m2g = pd.read_csv(CORR_DIR / "workflow_corr_m2g.csv")

all_correspondences = pd.concat([correspondences_m2a, correspondences_m2g], ignore_index=True)

len(all_correspondences)

31814

In [43]:
strategy = DataFusionStrategy('books_fusion_strategy')

strategy.add_attribute_fuser('title', longest_string)
strategy.add_attribute_fuser('author', longest_string)
strategy.add_attribute_fuser('publish_year', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('publisher', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('isbn_clean', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('price', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('page_count', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('rating', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('genres', union)

In [17]:
engine = DataFusionEngine(strategy, debug=True, debug_format='json',
                          debug_file= PIPELINE_DIR / "debug_fusion_ml_sn_blocker.jsonl")

fused_ml_snblocker = engine.run(
    datasets=[amazon_books, metabooks, goodreads],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False,
)

print(f'Fused rows: {len(fused_ml_snblocker):,}')
display(fused_ml_snblocker.head())

Fused rows: 30,185


,_id,_fusion_group_id,_fusion_sources,numratings,title,id,rating,price,publish_year,language,genres,isbn_clean,page_count,author,publisher,_fusion_confidence,_fusion_metadata,edition,bookformat
0,metabooks_401477,group_0,"[metabooks, amazon_books]",107,eyes wide shut a screenplay,metabooks_401477,4.4,62.18,1999.0,English,"[['Humor', 'Entertainment', 'Movies']]",0446676322,281.0,arthur schnitzler,Warner Books,0.842741,"{'numratings_rule': 'first_non_null', 'numrati...",NaN,NaN
1,amazon_123817,group_1,"[metabooks, amazon_books]",6,a secret history of time to come,amazon_123817,4.5,4.50,1979.0,English,None,0394501667,303.0,robie macauley,Knopf,0.727273,"{'title_rule': 'longest_string', 'title_source...",NaN,NaN
2,metabooks_347208,group_2,"[metabooks, amazon_books]",492,rest in pieces mrs murphy mysteries paperback,metabooks_347208,4.6,7.99,1993.0,English,"[['Mystery, Thriller', 'Suspense', 'Mystery']]",0553562398,384.0,sneaky pie brown,Bantam,0.833965,"{'numratings_rule': 'first_non_null', 'numrati...",NaN,NaN
3,amazon_107993,group_3,"[metabooks, amazon_books]",160,liberated parents liberated children your guid...,amazon_107993,4.8,6.99,1990.0,English,"[['Self-Help', 'Relationships']]",0380711346,272.0,adele faber,Perennial Currents,0.818182,"{'title_rule': 'longest_string', 'title_source...",NaN,NaN
4,metabooks_355604,group_4,"[metabooks, amazon_books]",32,the complete idiots guide to getting rich 2nd ...,metabooks_355604,4.7,NaN,1999.0,English,"[['Business', 'Money', 'Personal Finance']]",0028629523,368.0,larry waschka,Alpha Communications,0.734991,"{'numratings_rule': 'first_non_null', 'numrati...",NaN,NaN


In [18]:
fused_ml_snblocker.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30185 entries, 0 to 30184
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   _id                 30185 non-null  object 
 1   _fusion_group_id    30185 non-null  object 
 2   _fusion_sources     30185 non-null  object 
 3   numratings          30185 non-null  int64  
 4   title               30185 non-null  object 
 5   id                  30185 non-null  object 
 6   rating              30185 non-null  float64
 7   price               27975 non-null  float64
 8   publish_year        30171 non-null  float64
 9   language            30069 non-null  object 
 10  genres              28964 non-null  object 
 11  isbn_clean          30185 non-null  object 
 12  page_count          27473 non-null  float64
 13  author              30185 non-null  object 
 14  publisher           30185 non-null  object 
 15  _fusion_confidence  30185 non-null  float64
 16  _fus

In [19]:
fused_ml_snblocker.to_parquet(PIPELINE_DIR / "fused_ml_snblocker.parquet")

### Fusion Evaluation

In [80]:
fused_dataset = pd.read_parquet(PIPELINE_DIR / "fused_ml_snblocker.parquet")
fused_dataset["publish_year"] = fused_dataset["publish_year"].astype("Int16")
fused_dataset["page_count"] = fused_dataset["page_count"].astype("Int32")
golden_fused_dataset= pd.read_parquet(MLDS_DIR / "golden_fused_books.parquet")

In [120]:
from PyDI.fusion import tokenized_match, boolean_match,numeric_tolerance_match,set_equality_match

import numpy as np
import re, ast, numpy as np, pandas as pd

def _is_missing_value(v):
    if v is None:
        return True
    if isinstance(v, float) and np.isnan(v):
        return True
    if isinstance(v, (list, tuple, set, np.ndarray)):
        return len(v) == 0
    if isinstance(v, str):
        s = v.strip().lower()
        return s in ("", "nan", "none")
    return False

def array_set_equality_match(fused_value, gold_value) -> bool:
    """Case-insensitive set equality for strings, arrays, or stringified lists."""
    if _is_missing_value(fused_value) and _is_missing_value(gold_value):
        return True
    if _is_missing_value(fused_value) or _is_missing_value(gold_value):
        return False

    def to_clean_set(value):
        # unwrap 0-d or 1-element numpy arrays
        if isinstance(value, np.ndarray):
            # flatten and collapse 1-element arrays
            flat = value.flatten().tolist()
            if len(flat) == 1:
                value = flat[0]
            else:
                value = flat

        # parse stringified lists like "['Fiction','Drama']"
        if isinstance(value, str):
            s = value.strip()
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple, set, np.ndarray)):
                    value = parsed
                else:
                    # simple delimited string
                    value = re.split(r"[|,;/]", s)
            except Exception:
                # not parsable, split manually
                value = re.split(r"[|,;/]", s)

        # flatten iterable containers
        if isinstance(value, np.ndarray):
            value = value.tolist()
        if isinstance(value, (list, tuple, set)):
            items = []
            for v in value:
                # recursively parse if element looks like "['a','b']"
                if isinstance(v, str) and v.strip().startswith("["):
                    try:
                        inner = ast.literal_eval(v)
                        if isinstance(inner, (list, tuple, set)):
                            items.extend(inner)
                            continue
                    except Exception:
                        pass
                items.append(v)
            return {str(x).strip().lower() for x in items if not _is_missing_value(x)}

        # scalar fallback
        return {str(value).strip().lower()}

    fused_set = to_clean_set(fused_value)
    gold_set  = to_clean_set(gold_value)

    if not fused_set and not gold_set:
        return True
    return fused_set == gold_set



strategy.add_evaluation_function("title", tokenized_match)
strategy.add_evaluation_function("author", tokenized_match)
strategy.add_evaluation_function("publisher", tokenized_match)
strategy.add_evaluation_function("publish_year", numeric_tolerance_match)
strategy.add_evaluation_function("rating", numeric_tolerance_match)
strategy.add_evaluation_function("numratings", numeric_tolerance_match)
strategy.add_evaluation_function("price", numeric_tolerance_match)
strategy.add_evaluation_function("page_count", numeric_tolerance_match)
strategy.add_evaluation_function("genres", array_set_equality_match)





In [74]:
dupe_rows = fused_dataset[fused_dataset['isbn_clean'].duplicated(keep=False)]
dupe_rows.sort_values('isbn_clean')


,_id,_fusion_group_id,_fusion_sources,numratings,title,id,rating,price,publish_year,language,genres,isbn_clean,page_count,author,publisher,_fusion_confidence,_fusion_metadata,edition,bookformat
3948,metabooks_323815,group_3948,"[goodreads, metabooks]",1,donald ducks toy sailboat,metabooks_323815,5.0,7.98,1990.0,English,"[['Picture Books', 'Childrens', 'Animals', 'Fi...",0307021459,24.0,annie north bedford samuel armstrong adaptor,Golden Press,0.791084,"{'_id_inputs': [{'dataset': 'metabooks', 'reco...",nan,Paperback
8125,amazon_60391,group_8125,"[metabooks, amazon_books]",26,walt disneys donald ducks toy sailboat a littl...,amazon_60391,4.5,7.79,1993.0,English,"[[""Children's Books""]]",0307021459,NaN,annie north bedford,Western Pub. Co,0.727273,"{'_id_inputs': [{'dataset': 'amazon_books', 'r...",None,None
9291,metabooks_328776,group_9291,"[metabooks, amazon_books]",11,the corps semper fibk 1 corps paperback,metabooks_328776,4.6,6.57,1988.0,English,None,0515087491,NaN,w e b griffin,Jove Books,0.659674,"{'_id_inputs': [{'dataset': 'metabooks', 'reco...",None,None
26335,metabooks_530857,group_26335,"[goodreads, metabooks]",2692,semper fi the corps book 1,metabooks_530857,4.7,8.99,1986.0,English,"[['Fiction', 'Historical Fiction', 'Military F...",0515087491,352.0,w e b griffin,G.P. Putnam's Sons,0.800296,"{'_id_inputs': [{'dataset': 'metabooks', 'reco...",nan,Paperback


In [124]:
from PyDI.fusion import DataFusionEvaluator
fused_dataset.drop_duplicates(subset='isbn_clean', keep='first',inplace=True)
# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval.jsonl", debug_format="json")

# Evaluate the fused results against the gold standard
print("Evaluating fusion results against gold standard...")
evaluation_results = evaluator.evaluate(
    fused_df=fused_dataset,
    fused_id_column='isbn_clean',
    gold_df=golden_fused_dataset,
    gold_id_column='isbn_clean',
)

# Display evaluation metrics
print("\nFusion Evaluation Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

Evaluating fusion results against gold standard...

Fusion Evaluation Results:
  overall_accuracy: 0.637
  macro_accuracy: 0.638
  num_evaluated_records: 38828
  num_evaluated_attributes: 9
  total_evaluations: 342147
  total_correct: 217787
  rating_accuracy: 0.630
  rating_count: 38828
  genres_accuracy: 0.640
  genres_count: 37448
  page_count_accuracy: 0.678
  page_count_count: 35411
  numratings_accuracy: 0.690
  numratings_count: 38828
  title_accuracy: 0.499
  title_count: 38828
  publish_year_accuracy: 0.745
  publish_year_count: 38708
  publisher_accuracy: 0.341
  publisher_count: 38828
  author_accuracy: 0.760
  author_count: 38828
  price_accuracy: 0.756
  price_count: 36440

Overall Accuracy: 63.7%
